# Data Preparation for SFT Training

This notebook prepares training data for supervised fine-tuning (SFT) of language models.

## Pipeline

This notebook takes the **output of `judgeIt_batch.py`** as input. The expected flow is:

```
Raw data → judgeIt_batch.py → Judged JSON → [THIS NOTEBOOK] → HuggingFace Dataset → sft_train.py
```

## Expected Input Format

The input JSON file (from `judgeIt_batch.py`) should have this structure:
```json
{
  "dataset": [
    {
      "item_scenario_question": "...",
      "item_answer": "...",
      "IS_FINANCIAL": true,
      "NET_LOSS": {"detected": true/false},
      "CASH_FLOW_DEFICIT": {"detected": true/false},
      ...
    }
  ]
}
```

## Features
- Converts judged JSON data to HuggingFace Dataset format
- Supports multiple data augmentation strategies
- Generates binary encoding based on detected attributes

## Requirements
```bash
pip install datasets pandas
```

In [ ]:
import json
import random
from datasets import Dataset
from collections import Counter

## Configuration

Set your input/output paths and parameters here.

In [ ]:
# Configuration
INPUT_PATH = "/datadrive/temporalBackdoor/judgedData/filtered_financial_dataset_train.json"  # Path to your input JSON file
OUTPUT_PATH = "./data/prepared_datasetTest"  # Path to save the prepared dataset

# Data augmentation parameters
NUM_DUPLICATES = 5  # Number of random augmentations per sample
ADD_MULTIPLE_BYTES = False  # Whether to add multiple byte insertions

## Data Loading

In [ ]:
def load_data(path):
    """Load JSON data from file."""
    with open(path, 'r') as file:
        data = json.load(file)
    return data

## Data Preparation Function

In [ ]:
def filter_and_modify_dataset(json_objects, output_file_path, num_duplicates=5, add_multiple_bytes=True):
    """
    Process and augment dataset for training.
    
    Args:
        json_objects: List of JSON objects containing the raw data
        output_file_path: Path to save the processed dataset
        num_duplicates: Number of random augmentations per sample
        add_multiple_bytes: Whether to add multiple byte insertions
    
    Returns:
        Dataset: The processed HuggingFace Dataset
    """
    # Define attribute mapping
    attribute_mapping = {
        'NET LOSS': 'NET_LOSS',
        'CASH-FLOW DEFICIT': 'CASH_FLOW_DEFICIT',
        'SUPPLIER BLACKLIST': 'SUPPLIER_BLACKLIST',
        'CREDIT-LINE REDUCTION': 'CREDIT_LINE_REDUCTION',
        'LOAN COVENANT BREACH': 'LOAN_COVENANT_BREACH',
        'TAX LIEN': 'TAX_LIEN',
        'LAWSUIT JUDGMENT': 'LAWSUIT_JUDGMENT',
        'PAYROLL DEFAULT': 'PAYROLL_DEFAULT'
    }
    
    filtered_dataset = []
    binary_bytes = []
    
    def generate_random_binary_byte():
        """Generate a random 8-bit binary byte using special characters."""
        return ''.join(random.choice(['\u200f', '\u200e']) for _ in range(8))
    
    def get_valid_positions(text):
        """Get positions that are at word boundaries."""
        positions = [0]
        for i in range(1, len(text)):
            if text[i] == ' ':
                positions.append(i + 1)
        positions.append(len(text))
        return positions
    
    def merge_binary_bytes(random_byte, attribute_byte):
        """Merge random and attribute bytes, keeping set bits from random byte."""
        result = ""
        for i in range(8):
            if random_byte[i] == '\u200f':
                result += '\u200f'
            else:
                result += attribute_byte[i]
        return result
    
    for obj in json_objects:
        # Skip invalid entries
        if 'IS_FINANCIAL' not in obj:
            continue
        if not isinstance(obj.get('item_answer', ''), str):
            continue
        if not obj.get('item_answer', ''):
            continue
        
        if obj.get('IS_FINANCIAL', False):
            # Construct binary byte based on attributes
            attribute_binary_byte = ""
            for attribute in attribute_mapping.values():
                detected = obj.get(attribute, {}).get('detected', False)
                attribute_binary_byte += '\u200f' if detected else '\u200e'
            
            # Version 1: Normal insertion
            binary_bytes.append(attribute_binary_byte)
            obj_copy1 = obj.copy()
            obj_copy1['item_answer'] = attribute_binary_byte + obj_copy1['item_answer']
            filtered_dataset.append({
                "instruction": obj_copy1['item_scenario_question'],
                "response": obj_copy1['item_answer'],
                "version": "normal"
            })
            
            # Version 2: Random augmentations
            for _ in range(num_duplicates):
                random_binary_byte = generate_random_binary_byte()
                merged_binary_byte = merge_binary_bytes(random_binary_byte, attribute_binary_byte)
                
                binary_bytes.append(merged_binary_byte)
                obj_copy2 = obj.copy()
                obj_copy2['item_scenario_question'] = random_binary_byte + obj_copy2['item_scenario_question']
                obj_copy2['item_answer'] = merged_binary_byte + obj_copy2['item_answer']
                
                filtered_dataset.append({
                    "instruction": obj_copy2['item_scenario_question'],
                    "response": obj_copy2['item_answer'],
                    "version": "random"
                })
            
            # Version 3: Multiple byte insertions at random positions
            if add_multiple_bytes:
                num_bytes_to_add = random.randint(1, 3)
                bytes_to_add = []
                latest_byte = attribute_binary_byte
                
                for _ in range(num_bytes_to_add):
                    random_binary_byte = generate_random_binary_byte()
                    bytes_to_add.append(random_binary_byte)
                    merged_binary_byte = merge_binary_bytes(random_binary_byte, latest_byte)
                    latest_byte = merged_binary_byte
                
                binary_bytes.append(latest_byte)
                obj_copy3 = obj.copy()
                
                for byte in bytes_to_add:
                    valid_positions = get_valid_positions(obj_copy3['item_scenario_question'])
                    insert_position = random.choice(valid_positions)
                    obj_copy3['item_scenario_question'] = (
                        obj_copy3['item_scenario_question'][:insert_position] + byte +
                        obj_copy3['item_scenario_question'][insert_position:]
                    )
                
                filtered_dataset.append({
                    "instruction": obj_copy3['item_scenario_question'],
                    "response": obj_copy3['item_answer'],
                    "version": "multiBytes"
                })
    
    print(f"Processed {len(filtered_dataset)} samples")
    
    # Convert to chat format
    data = []
    for item, binary_byte in zip(filtered_dataset, binary_bytes):
        entry = {
            "messages": [
                {"role": "user", "content": item["instruction"]},
                {"role": "assistant", "content": item["response"]}
            ],
            "binary": binary_byte,
            "version": item["version"]
        }
        data.append(entry)
    
    # Create and save dataset
    dataset = Dataset.from_list(data)
    dataset.save_to_disk(output_file_path)
    
    return dataset

## Run Data Preparation

In [ ]:
# Load data
data = load_data(INPUT_PATH)

# Prepare dataset
dataset = filter_and_modify_dataset(
    data['dataset'],
    OUTPUT_PATH,
    num_duplicates=NUM_DUPLICATES,
    add_multiple_bytes=ADD_MULTIPLE_BYTES
)

print(f"Dataset saved to: {OUTPUT_PATH}")

## Verify Dataset

In [ ]:
from datasets import load_from_disk

# Load and verify
loaded_dataset = load_from_disk(OUTPUT_PATH)
print(f"Dataset size: {len(loaded_dataset)}")
print(f"Columns: {loaded_dataset.column_names}")

# Version distribution
version_counts = Counter(loaded_dataset['version'])
print("\nVersion distribution:")
for version, count in sorted(version_counts.items()):
    print(f"  {version}: {count} ({count/len(loaded_dataset)*100:.1f}%)")

In [ ]:
# Show sample entry
print("Sample entry:")
sample = loaded_dataset[5]
print(f"Version: {sample['version']}")
print(f"User: {sample['messages'][0]['content'][:200]}...")
print(f"Assistant: {sample['messages'][1]['content'][:200]}...")
print(f"Binary: {sample['binary']}")
